<a href="https://colab.research.google.com/github/Blqcky23/bigquery-ecommerce-financial-analysis/blob/main/financial_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd


In [16]:
# Authenticate user session for Google Cloud access
auth.authenticate_user()

In [22]:
PROJECT_ID = "PASTE_YOUR_PROJECT_ID_HERE"
client = bigquery.Client(project=PROJECT_ID)

In [15]:
# Query aggregated financial metrics per product category
query = """
SELECT
    p.category,
    ROUND(SUM(oi.sale_price), 2) AS total_revenue,
    ROUND(SUM(p.cost), 2) AS total_cost,
    ROUND(SUM(oi.sale_price - p.cost), 2) AS gross_profit
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
JOIN `bigquery-public-data.thelook_ecommerce.products` p ON oi.product_id = p.id
WHERE oi.status = 'Complete'
GROUP BY p.category
ORDER BY gross_profit DESC
"""


In [9]:
df = client.query(query).to_dataframe()


In [17]:
# --- FINANCIAL DATA VALIDATION CHECKS ---
# 1. Ensure no missing values exist in the aggregated financial dataset

assert not df.isnull().values.any(), "Błąd: Wykryto puste wartości (NaN)!"

# 2. Verify all category revenues are strictly positive
assert (df["total_revenue"] > 0).all(), "Błąd: Przychód musi być dodatni!"

In [21]:
# 3. Cross-check aggregated revenue against raw table sum to prevent join duplication
check_query = """
SELECT ROUND(SUM(sale_price), 2) AS raw_total
FROM `bigquery-public-data.thelook_ecommerce.order_items`
WHERE status = 'Complete'
"""

raw_total = client.query(check_query).to_dataframe()["raw_total"].iloc[0]
aggregated_total = round(df["total_revenue"].sum(), 2)

assert abs(raw_total - aggregated_total) < 0.01, f"Reconciliation Mismatch: Raw total {raw_total} != Aggregated  {aggregated_total}"
print(f"Validation Passed! Verified Total Revenue: ${aggregated_total:,.2f}")
print(df.head())

Validation Passed! Verified Total Revenue: $2,685,620.84
              category  total_revenue  total_cost  gross_profit
0    Outerwear & Coats      326920.19   145639.87     181280.32
1                Jeans      300843.44   161127.19     139716.25
2             Sweaters      203605.02    98135.63     105469.39
3  Suits & Sport Coats      162041.43    65181.14      96860.29
4                 Swim      160403.85    81879.05      78524.80
